# Wah-Wah matériel sur PYNQ

Ce notebook charge l'overlay `wahwah.bit`, configure l'IP HLS, envoie le son au FPGA avec le canal **MM2S** du DMA et récupère le résultat avec le canal **S2MM**.

> **Important :** le code HLS actuel n'implémente pas exactement le même filtre que `wahwahSoft.ipynb`.  
> Le notebook logiciel utilise un filtre passe-bande biquad avec le paramètre `Q`, tandis que l'IP HLS actuelle utilise un filtre RC du premier ordre dont la fréquence de coupure varie avec le LFO. Le chargement du fichier WAV, les paramètres, la sauvegarde et les tracés reprennent toutefois l'organisation du tutoriel logiciel.

In [ ]:
from pynq import Overlay, allocate, PL
import numpy as np
import scipy.io.wavfile as wav
import matplotlib.pyplot as plt
import struct
import time
from pathlib import Path

BITSTREAM = "wahwah.bit"
INPUT_WAV = "PinkPanther30.wav"
OUTPUT_WAV = "PinkPanther30_hardware.wav"

MIN_FREQ = 500.0
MAX_FREQ = 3000.0
LFO_FREQ = 1.0

# Taille modérée pour éviter de réserver un très gros buffer contigu.
CHUNK_SAMPLES = 262_144

## 1. Chargement de l'overlay

In [ ]:
PL.reset()
overlay = Overlay(BITSTREAM)

print("Overlay chargé.")
print("Blocs IP détectés :", list(overlay.ip_dict.keys()))

## 2. Détection de l'IP Wah-Wah et du DMA

In [ ]:
def find_ip_name(ip_dict, required_words):
    required_words = [word.lower() for word in required_words]
    matches = [
        name for name in ip_dict
        if all(word in name.lower() for word in required_words)
    ]
    if not matches:
        raise RuntimeError(
            f"Bloc introuvable avec les mots {required_words}. "
            f"Blocs disponibles : {list(ip_dict.keys())}"
        )
    return matches[0]

wahwah_name = find_ip_name(overlay.ip_dict, ["wah", "filter"])
dma_name = find_ip_name(overlay.ip_dict, ["dma"])

wahwah_filter = getattr(overlay, wahwah_name)
dma = getattr(overlay, dma_name)

dma_send = dma.sendchannel   # MM2S : DDR -> IP
dma_recv = dma.recvchannel   # S2MM : IP -> DDR

print("IP Wah-Wah :", wahwah_name)
print("AXI DMA     :", dma_name)
print("Registres   :", overlay.ip_dict[wahwah_name].get("registers", {}))

## 3. Configuration des registres AXI-Lite

In [ ]:
def register_offset(ip_name, parameter):
    registers = overlay.ip_dict[ip_name].get("registers", {})
    if parameter not in registers:
        raise KeyError(
            f"Registre '{parameter}' absent. "
            f"Registres disponibles : {list(registers.keys())}"
        )
    return registers[parameter]["address_offset"]

def float_to_u32(value):
    return struct.unpack("<I", struct.pack("<f", float(value)))[0]

def u32_to_float(value):
    return struct.unpack("<f", struct.pack("<I", int(value)))[0]

CONTROL_REGISTER = 0x00
SAMPLE_RATE_REGISTER = register_offset(wahwah_name, "sample_rate")
MIN_FREQ_REGISTER = register_offset(wahwah_name, "min_freq")
MAX_FREQ_REGISTER = register_offset(wahwah_name, "max_freq")
LFO_FREQ_REGISTER = register_offset(wahwah_name, "lfo_freq")

def configure_filter(sample_rate, min_freq, max_freq, lfo_freq):
    if sample_rate <= 0:
        raise ValueError("sample_rate doit être strictement positif.")
    if not (0 < min_freq < max_freq < sample_rate / 2):
        raise ValueError(
            "Il faut respecter : 0 < min_freq < max_freq < sample_rate/2."
        )
    if lfo_freq <= 0:
        raise ValueError("lfo_freq doit être strictement positif.")

    wahwah_filter.write(SAMPLE_RATE_REGISTER, int(sample_rate))
    wahwah_filter.write(MIN_FREQ_REGISTER, float_to_u32(min_freq))
    wahwah_filter.write(MAX_FREQ_REGISTER, float_to_u32(max_freq))
    wahwah_filter.write(LFO_FREQ_REGISTER, float_to_u32(lfo_freq))

    print("Paramètres écrits dans l'IP :")
    print(" sample_rate =", wahwah_filter.read(SAMPLE_RATE_REGISTER))
    print(" min_freq   =", u32_to_float(wahwah_filter.read(MIN_FREQ_REGISTER)))
    print(" max_freq   =", u32_to_float(wahwah_filter.read(MAX_FREQ_REGISTER)))
    print(" lfo_freq   =", u32_to_float(wahwah_filter.read(LFO_FREQ_REGISTER)))

## 4. Lecture et préparation du fichier audio

Le FPGA reçoit ici des échantillons entiers signés sur 32 bits. Un fichier stéréo est converti en mono, comme dans le notebook logiciel.

In [ ]:
input_path = Path(INPUT_WAV)
if not input_path.exists():
    raise FileNotFoundError(
        f"Le fichier '{INPUT_WAV}' doit être placé dans le même dossier que le notebook."
    )

sample_rate, input_signal = wav.read(input_path)
original_dtype = input_signal.dtype

if input_signal.ndim == 2:
    # Moyenne en int64 pour éviter un dépassement pendant l'addition.
    input_signal = np.rint(
        input_signal.astype(np.int64).mean(axis=1)
    )

if np.issubdtype(original_dtype, np.floating):
    # WAV flottant normalement compris entre -1 et 1.
    input_int32 = np.rint(
        np.clip(input_signal, -1.0, 1.0) * 32767.0
    ).astype(np.int32)
elif original_dtype == np.int16:
    input_int32 = input_signal.astype(np.int32)
elif original_dtype == np.int32:
    input_int32 = input_signal.astype(np.int32)
elif original_dtype == np.uint8:
    input_int32 = (input_signal.astype(np.int32) - 128) << 8
else:
    raise TypeError(f"Format WAV non pris en charge : {original_dtype}")

configure_filter(sample_rate, MIN_FREQ, MAX_FREQ, LFO_FREQ)

duration = len(input_int32) / sample_rate
print("Fréquence d'échantillonnage :", sample_rate, "Hz")
print("Nombre d'échantillons       :", len(input_int32))
print("Durée                        :", round(duration, 3), "s")
print("Type WAV d'origine           :", original_dtype)

## 5. Traitement matériel par blocs

Les états internes du filtre et le temps du LFO sont déclarés `static` dans le code HLS. Ils sont donc conservés entre deux blocs tant que l'overlay n'est pas rechargé.

In [ ]:
def process_chunk_fpga(chunk):
    chunk = np.ascontiguousarray(chunk, dtype=np.int32)

    in_buffer = allocate(shape=(len(chunk),), dtype=np.int32)
    out_buffer = allocate(shape=(len(chunk),), dtype=np.int32)

    try:
        in_buffer[:] = chunk
        in_buffer.flush()

        # Prépare les deux directions du DMA avant de lancer l'IP.
        dma_recv.transfer(out_buffer)
        dma_send.transfer(in_buffer)

        # ap_start = 1
        wahwah_filter.write(CONTROL_REGISTER, 0x01)

        dma_send.wait()
        dma_recv.wait()
        out_buffer.invalidate()

        return np.array(out_buffer, dtype=np.int32, copy=True)
    finally:
        in_buffer.freebuffer()
        out_buffer.freebuffer()


def process_fpga(signal, chunk_samples=CHUNK_SAMPLES):
    signal = np.ascontiguousarray(signal, dtype=np.int32)
    output = np.empty_like(signal)

    total = len(signal)
    for start in range(0, total, chunk_samples):
        stop = min(start + chunk_samples, total)
        output[start:stop] = process_chunk_fpga(signal[start:stop])

        percentage = 100.0 * stop / total
        print(
            f"\rTraitement : {stop}/{total} échantillons "
            f"({percentage:5.1f} %)",
            end=""
        )

    print()
    return output

In [ ]:
start_time = time.time()
filtered_int32 = process_fpga(input_int32)
processing_time = time.time() - start_time

print(f"Traitement FPGA terminé en {processing_time:.3f} s.")
print(
    f"Débit moyen : "
    f"{len(input_int32) / max(processing_time, 1e-9):,.0f} échantillons/s"
)

## 6. Sauvegarde du résultat

In [ ]:
# L'IP produit des valeurs entières dans la même échelle que son entrée.
filtered_int16 = np.clip(
    filtered_int32,
    np.iinfo(np.int16).min,
    np.iinfo(np.int16).max
).astype(np.int16)

wav.write(OUTPUT_WAV, sample_rate, filtered_int16)
print(f"Audio matériel sauvegardé dans : {OUTPUT_WAV}")

## 7. Affichage du signal original et filtré

In [ ]:
time_axis = np.arange(len(input_int32)) / sample_rate

# Affichage limité pour garder un tracé fluide sur les longs fichiers.
display_seconds = min(5.0, duration)
display_samples = int(display_seconds * sample_rate)

plt.figure(figsize=(15, 5))
plt.plot(
    time_axis[:display_samples],
    input_int32[:display_samples],
    label="Signal original",
    alpha=0.7
)
plt.plot(
    time_axis[:display_samples],
    filtered_int32[:display_samples],
    label="Signal filtré par le FPGA",
    alpha=0.7
)
plt.xlabel("Temps (s)")
plt.ylabel("Amplitude")
plt.title(
    f"Wah-Wah matériel — {display_seconds:.1f} premières secondes "
    f"— traitement total : {processing_time:.3f} s"
)
plt.legend()
plt.grid(True)
plt.show()

## 8. Comparaison simple des amplitudes

In [ ]:
print("Entrée :")
print(" min =", int(input_int32.min()))
print(" max =", int(input_int32.max()))
print(" RMS =", float(np.sqrt(np.mean(input_int32.astype(np.float64) ** 2))))

print("\nSortie FPGA :")
print(" min =", int(filtered_int32.min()))
print(" max =", int(filtered_int32.max()))
print(" RMS =", float(np.sqrt(np.mean(filtered_int32.astype(np.float64) ** 2))))